# WGCNA: Co-expression Network Analysis (R notebook)
### Which genes move together, and do those modules track disease or just the lab they came from?

**What this notebook does.**

1. Loads the batch-corrected merged expression matrix and sample sheet from the Python run.
2. Builds a **signed co-expression network** on the 4,000 most variable genes and cuts it into **modules**.
3. Relates each module to **disease traits** (HC, JDM, interferonopathy) *and* to **batch traits** (which dataset/file a sample came from).
4. Runs follow-up checks to decide which modules are biology and which are leftover batch: eigengene vs PCA, marker genes for blood cell types, expression level, and a **gene-set test (mroast)**.

**How to read it.** Each part has a 🎯 **Goal**, a 💡 **Why**, your code unchanged, and a 👀 **Checkpoint**. Plots your code saves as PNG are shown inline.

**Roadmap**

| Part | Step | Output |
|---|---|---|
| 1 | Configuration | settings, thresholds, IFN gene list |
| 2 | Packages | WGCNA installed and loaded |
| 3 | Load & filter | `datExpr` (samples × genes) |
| 4 | Traits | `traits` (disease + batch) |
| 5 | Sample outlier view | `wgcna_sample_tree.png` |
| 6 | Soft threshold | `power` |
| 7 | Network & modules | `net`, `moduleColors`, `MEs` |
| 8 | Module–trait table | `mtc`, `mtp`, `summ` |
| 9 | Gene tables & heatmap | `wgcna_gene_modules.csv`, heatmap |
| 10 | Environment check | what is installed |
| 11 | Follow-up analyses | is a module biology or batch? |

<div class="alert alert-warning">
<b>The central problem this notebook is built around.</b> Your datasets are <i>confounded with disease</i>: JDM exists only in the JDM-lab data, while CANDLE/SAVI exist only in the Goldbach file. A module that correlates with "JDM" might really just correlate with "came from the JDM lab". That is why batch traits sit next to disease traits, why modules get a <code>source_flag</code>, and why the one <b>unconfounded</b> contrast (JDM vs HC <i>inside</i> the JDM-lab data) gets its own column.
</div>

<div class="alert alert-info">
<b>Kernel.</b> This is an <b>R</b> notebook. Make sure the kernel is R (the file is saved with the R kernel already selected). The first run installs WGCNA if missing and can take 10+ minutes.
</div>

---
# Part 1 · Configuration

🎯 **Goal:** every tunable choice in one cell.

💡 **Why each setting matters:**

| Setting | What it controls | Rationale |
|---|---|---|
| `TOP_N = 4000` | genes used | WGCNA's correlations are dominated by noise in low-variance genes; using the most variable genes keeps the network meaningful *and* the computation feasible |
| `MIN_MODULE = 30` | smallest module | avoids tiny modules that are just a few correlated genes |
| `NETWORK = "signed"` | signed network | in a *signed* network, positively and negatively correlated genes are **not** grouped together, which gives biologically cleaner modules (e.g. all interferon genes go up together) |
| `MERGE_CUT = 0.25` | module merging | modules whose eigengenes correlate > 0.75 (1 − 0.25) are merged as redundant |
| `R2_TARGET = 0.80` | scale-free fit | the soft-threshold power is accepted only if the network looks scale-free (see Part 6) |
| `FALLBACK_POWER = 12` | backup power | the WGCNA recommendation for signed networks with more than 40 samples |
| `BATCH_FLAG_R = 0.5` | batch flag | any module with \|r\| > 0.5 to a source trait is flagged as possibly batch-driven |
| `INTERFERONOPATHY` | trait grouping | which diagnoses are pooled into one "Interferonopathy" class |
| `IFN_GENES` | marker list | lets each module report how many canonical interferon-stimulated genes it contains |

In [ ]:
# =============================================================================
# WGCNA, ONE CELL for an R notebook (R kernel).
# Input : the batch-corrected merged matrix + merged_samples.csv from the Python run.
# Output: /mnt/file-systems/rnaseq_out/wgcna/  (CSV tables + PNG plots)
#   signed network, top-variance genes, power from the scale-free fit,
#   module-trait table with BATCH traits (source dataset / file) next to the
#   disease traits so batch-driven modules can be recognised.
# First run installs WGCNA if missing (can take 10+ minutes).
# =============================================================================
options(stringsAsFactors = FALSE)

# ---- configuration ----------------------------------------------------------
DIR        <- "/mnt/file-systems/rnaseq_out"
EXPR_FILE  <- file.path(DIR, "merged_log2_RPKM_QN_batchcorrected.csv")   # genes x samples
META_FILE  <- file.path(DIR, "merged_samples.csv")
OUT        <- file.path(DIR, "wgcna")
TOP_N      <- 4000        # most variable genes kept (of ~11,000)
MIN_MODULE <- 30          # smallest allowed module
NETWORK    <- "signed"
MERGE_CUT  <- 0.25        # modules whose eigengenes correlate > 0.75 are merged
R2_TARGET  <- 0.80        # scale-free fit needed to accept a soft-threshold power
FALLBACK_POWER <- 12      # used only if no power reaches R2_TARGET (signed, n > 40)
BATCH_FLAG_R   <- 0.5     # |r| with a source trait above this => flag the module
INTERFERONOPATHY <- c("AGS", "CANDLE", "CANDLE-like", "SAVI", "Undifferentiated Interferonopathy")
IFN_GENES  <- c("IFI44L","IFI44","IFIT1","IFIT3","ISG15","MX1","OAS1","RSAD2",
                "IFI27","USP18","OASL","HERC5","IFI6","LY6E","SIGLEC1","EPSTI1")
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

### 1.1 A small display helper *(added for this walkthrough)*

💡 **Why:** your code saves plots as PNG files on the mount, which you can't see from inside the notebook. This helper shows a saved PNG inline using `IRdisplay` (installed with the R kernel). If it isn't available, it just prints the file path. It is used only by the 👀 Checkpoints and does not touch your analysis.

In [ ]:
show_png <- function(f, width = 900) {
  if (!file.exists(f)) { cat("missing:", f, "\n"); return(invisible()) }
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = f, width = width)
  } else cat("saved to", f, "(IRdisplay not available to show it here)\n")
}

---
# Part 2 · Packages

🎯 **Goal:** make sure WGCNA and its Bioconductor dependencies are installed, then load it.

💡 **Why the install is so involved:** WGCNA depends on Bioconductor packages (`impute`, `preprocessCore`, `GO.db`, `AnnotationDbi`) that plain `install.packages()` can't fetch. The cell installs those through `BiocManager` first, then WGCNA, and only if WGCNA is missing, so later runs skip all of it. `allowWGCNAThreads(4)` enables parallel correlation, wrapped in `try()` so a restricted environment doesn't stop the run.

In [ ]:
# ---- packages ---------------------------------------------------------------
if (!requireNamespace("WGCNA", quietly = TRUE)) {
  message("installing WGCNA (first run only) ...")
  if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install(c("impute", "preprocessCore", "GO.db", "AnnotationDbi"),
                       ask = FALSE, update = FALSE)
  install.packages("WGCNA", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(WGCNA))
try(allowWGCNAThreads(4), silent = TRUE)

---
# Part 3 · Load, clean, and keep the most variable genes

🎯 **Goal:** `datExpr`, a **samples × genes** matrix (the orientation WGCNA requires), restricted to informative genes.

💡 **Why each guard:**

- **`stopifnot(file.exists…)`**: fail immediately and clearly if an input is missing.
- **Stale-file warning:** the batch-corrected CSV must be *newer* than the uncorrected one. If it's older, you'd be analysing a leftover from a previous run and the warning tells you to re-run the batch-correction step.
- **`match()` on sample IDs:** aligns the sample sheet to the matrix columns by name, so a different row order in the two files can't silently mislabel samples.
- **`goodSamplesGenes`:** removes genes/samples with too many missing or zero-variance values, which would break the correlation calculation.
- **Top-variance filter:** ranks genes by variance and keeps the top `TOP_N`. Low-variance genes contribute mostly noise to a correlation network.

In [ ]:
# ---- load -------------------------------------------------------------------
stopifnot(file.exists(EXPR_FILE), file.exists(META_FILE))
if (file.mtime(EXPR_FILE) < file.mtime(file.path(DIR, "merged_log2_RPKM_QN.csv")))
  warning("The batch-corrected file is OLDER than the main run's output. ",
          "Re-run the batch-correction cell first.")

expr <- read.csv(EXPR_FILE, row.names = 1, check.names = FALSE)          # genes x samples
meta <- read.csv(META_FILE, check.names = FALSE, stringsAsFactors = FALSE)
meta <- meta[match(colnames(expr), meta$sample_id), ]
stopifnot(!anyNA(meta$sample_id))
datExpr <- t(as.matrix(expr))                                            # samples x genes
cat("matrix:", nrow(datExpr), "samples x", ncol(datExpr), "genes\n")

gsg <- goodSamplesGenes(datExpr, verbose = 0)
if (!gsg$allOK) {
  cat("removing", sum(!gsg$goodGenes), "genes and", sum(!gsg$goodSamples), "samples with too many missing/zero-variance values\n")
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]
  meta    <- meta[gsg$goodSamples, ]
}
v <- apply(datExpr, 2, var)
datExpr <- datExpr[, order(v, decreasing = TRUE)[seq_len(min(TOP_N, ncol(datExpr)))]]
cat("kept the", ncol(datExpr), "most variable genes\n")

---
# Part 4 · Traits: disease *and* batch

🎯 **Goal:** a numeric `traits` table (samples × traits) to correlate module eigengenes against.

💡 **Why batch traits are included.** WGCNA correlates modules with numeric traits, so each category becomes a 0/1 column:

| Trait type | Columns | Why |
|---|---|---|
| **Disease** | `HC`, `JDM`, `Interferonopathy` | the biology you care about |
| **Demographic** | `age_yrs`, `male` | covariates; missing values stay `NA` and are handled per trait |
| **Batch (source)** | `src_jdm_lab`, `src_goldbach`, `src_hc_workbook` | which of three origins each sample came from |

A module strongly correlated with a *source* trait is suspicious: it may reflect lab rather than disease. Putting both kinds side by side makes this visible. Sources are derived from the dataset label and the source file name (anything not from the JDM lab and not a Goldbach file must be the HC workbook).

In [ ]:
# ---- traits: disease classes + batch (source) traits ------------------------
meta$class <- ifelse(meta$group == "HC", "HC",
              ifelse(meta$group == "JDM", "JDM",
              ifelse(meta$group %in% INTERFERONOPATHY, "Interferonopathy", NA)))
src3 <- ifelse(meta$dataset == "jdm_lab", "jdm_lab",
        ifelse(grepl("Goldbach", meta$source_file, ignore.case = TRUE), "goldbach", "hc_workbook"))
s1 <- tolower(substr(trimws(as.character(meta$sex)), 1, 1))
traits <- data.frame(
  HC               = as.numeric(meta$class == "HC"),
  JDM              = as.numeric(meta$class == "JDM"),
  Interferonopathy = as.numeric(meta$class == "Interferonopathy"),
  age_yrs          = suppressWarnings(as.numeric(meta$age_yrs)),
  male             = ifelse(s1 == "m", 1, ifelse(s1 == "f", 0, NA)),
  src_jdm_lab      = as.numeric(src3 == "jdm_lab"),
  src_goldbach     = as.numeric(src3 == "goldbach"),
  src_hc_workbook  = as.numeric(src3 == "hc_workbook"))
rownames(traits) <- meta$sample_id
BATCH_TRAITS   <- c("src_jdm_lab", "src_goldbach", "src_hc_workbook")
DISEASE_TRAITS <- c("HC", "JDM", "Interferonopathy")
cat("\nsamples per class and source:\n")
print(table(class = meta$class, source = src3, useNA = "ifany"))

### 👀 Checkpoint 4: how confounded is the design?

**Read the table above.** Look for **empty cells**. If JDM appears only under `jdm_lab` and Interferonopathy only under `goldbach`, then those disease traits *cannot* be separated from source by any correlation, which is exactly why Part 8 adds the within-dataset contrast. HC is the only class present in more than one source, so it is the bridge between them.

In [ ]:
library(stats)
tb <- table(class = meta$class, source = src3, useNA = "ifany")
op <- par(mar = c(4, 9, 3, 2))
barplot(t(tb), horiz = TRUE, las = 1, legend.text = colnames(tb),
        args.legend = list(x = "bottomright", cex = .8),
        main = "Samples per class, stacked by source", xlab = "samples")
par(op)

---
# Part 5 · Sample outlier view

🎯 **Goal:** a dendrogram of samples to spot anything that clusters far from everyone else.

💡 **Why:** WGCNA is sensitive to outlier samples, because a single aberrant sample can create a module on its own. Hierarchical clustering on Euclidean distance with average linkage is the standard visual screen. Labels are hidden (`labels = FALSE`) because with many samples they're unreadable; you're looking for **long branches** that split off alone.

In [ ]:
# ---- sample outlier view ----------------------------------------------------
png(file.path(OUT, "wgcna_sample_tree.png"), width = 1600, height = 700, res = 120)
plot(hclust(dist(datExpr), method = "average"), labels = FALSE, main = "Sample clustering", sub = "", xlab = "")
dev.off()

### 👀 Checkpoint 5

In [ ]:
show_png(file.path(OUT, "wgcna_sample_tree.png"))

---
# Part 6 · Choosing the soft-threshold power

🎯 **Goal:** pick the exponent β that turns raw correlations into a network.

💡 **Why a power at all.** WGCNA builds a network by raising each gene-pair correlation to a power: `connection = |cor|^β` (for signed: `((1 + cor)/2)^β`). A higher power pushes weak correlations towards zero and keeps only strong ones. The aim is a **scale-free topology**, where a few genes are highly connected "hubs" and most have few connections, as in most biological networks.

**How the choice is made.** `pickSoftThreshold` tests a range of powers and, for each, reports (a) how well the network fits a scale-free model (R²) and (b) its mean connectivity. The smallest power with R² ≥ `R2_TARGET` is taken, because larger powers shrink connectivity and make the network sparser than necessary. If **no** power reaches the target (common with many samples or strong batch structure), it falls back to the WGCNA-recommended default for signed networks and **says so** in the output.

In [ ]:
# ---- soft threshold ---------------------------------------------------------
powers <- c(1:10, seq(12, 20, by = 2))
sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = NETWORK, verbose = 0)
fit <- sft$fitIndices
write.csv(fit, file.path(OUT, "wgcna_soft_threshold.csv"), row.names = FALSE)
power <- sft$powerEstimate
if (is.na(power)) {
  power <- FALLBACK_POWER
  cat("\nno power reached R^2 >=", R2_TARGET, "- using fallback power", power, "\n")
} else cat("\nchosen soft-threshold power:", power, "\n")
png(file.path(OUT, "wgcna_soft_threshold.png"), width = 1400, height = 600, res = 120)
par(mfrow = c(1, 2))
plot(fit$Power, -sign(fit$slope) * fit$SFT.R.sq, type = "n", xlab = "power", ylab = "scale-free fit R^2")
text(fit$Power, -sign(fit$slope) * fit$SFT.R.sq, labels = fit$Power, col = "red")
abline(h = R2_TARGET, col = "red")
plot(fit$Power, fit$mean.k., type = "n", xlab = "power", ylab = "mean connectivity")
text(fit$Power, fit$mean.k., labels = fit$Power, col = "red")
dev.off()

### 👀 Checkpoint 6: is the power sensible?

**Left:** R² against power, with the red line at your target. You want the curve to cross it and then plateau. **Right:** mean connectivity, which should fall as power rises but not collapse to ~0. If the left curve never reaches the line, the fallback power was used, so treat module structure with extra caution.

In [ ]:
show_png(file.path(OUT, "wgcna_soft_threshold.png"), width = 1000)

---
# Part 7 · Build the network and find modules

🎯 **Goal:** assign every gene to a **module** (a set of co-expressed genes) and summarise each module with an **eigengene**.

💡 **Why each choice:**

- **`blockwiseModules`** does the whole pipeline: correlation → adjacency → **Topological Overlap Matrix (TOM)** → hierarchical clustering → dynamic tree cut. TOM measures not just whether two genes correlate but whether they share *neighbours*, which makes modules more robust to noise.
- **`maxBlockSize = ncol(datExpr)`** forces one block (all genes together). Splitting into blocks is only needed for tens of thousands of genes; with 4,000, one block avoids block-boundary artifacts.
- **`mergeCutHeight`** merges modules whose eigengenes are too similar.
- **`numericLabels = TRUE`** returns module numbers, then `labels2colors` converts them to the familiar colour names. **Colour names are assigned by module size**, so they can change if the data or settings change (see the warning in Part 11).
- **`grey`** is the "unassigned" bin, and is excluded from all downstream tables.
- **Module eigengene (ME)** = the first principal component of a module's genes: one number per sample summarising that module's overall activity. Everything downstream correlates *eigengenes*, not individual genes.

The `cor <- WGCNA::cor` / `cor <- stats::cor` pair is a known WGCNA workaround: WGCNA's `cor` must shadow base R's `cor` while `blockwiseModules` runs, then is restored.

In [ ]:
# ---- network and modules ----------------------------------------------------
cor <- WGCNA::cor                                   # required while blockwiseModules runs
net <- blockwiseModules(datExpr, power = power, networkType = NETWORK, TOMType = NETWORK,
                        minModuleSize = MIN_MODULE, mergeCutHeight = MERGE_CUT,
                        reassignThreshold = 0, pamRespectsDendro = FALSE,
                        numericLabels = TRUE, saveTOMs = FALSE,
                        maxBlockSize = ncol(datExpr), verbose = 0)
cor <- stats::cor
moduleColors <- labels2colors(net$colors)
cat("\nmodules found (grey = unassigned):\n"); print(sort(table(moduleColors), decreasing = TRUE))

### 7.1 Dendrogram with module colours and eigengenes

In [ ]:
png(file.path(OUT, "wgcna_dendrogram.png"), width = 1600, height = 700, res = 120)
tryCatch(plotDendroAndColors(net$dendrograms[[1]], moduleColors[net$blockGenes[[1]]],
                             "Module colors", dendroLabels = FALSE, hang = 0.03,
                             addGuide = TRUE, guideHang = 0.05),
         error = function(e) plot.new())
dev.off()

MEs <- orderMEs(moduleEigengenes(datExpr, colors = moduleColors)$eigengenes)
MEs <- MEs[, colnames(MEs) != "MEgrey", drop = FALSE]
stopifnot(ncol(MEs) > 0)

### 👀 Checkpoint 7: the modules

The colour bar under the dendrogram shows each gene's module. Healthy modules appear as solid colour blocks. A large **grey** share, or very fragmented colours, means weak co-expression structure.

In [ ]:
show_png(file.path(OUT, "wgcna_dendrogram.png"), width = 1000)

---
# Part 8 · Module–trait relationships (the heart of the analysis)

🎯 **Goal:** for every module, how strongly does its eigengene correlate with each trait, and is that disease or batch?

💡 **Why the details:**

- **Per-trait sample size.** Traits like `age_yrs` and `male` have missing values, so the correlation uses `pairwise.complete.obs` and the p-value uses **each trait's own n**, not the total sample count. Using the wrong n would overstate significance for traits with many `NA`s.
- **Student p-values by hand.** `t = r·√((n−2)/(1−r²))`, `p = 2·P(T>|t|)` with `n−2` degrees of freedom. `pmax(1−r², 1e-12)` prevents division by zero when |r| = 1.
- **The unconfounded contrast.** Because disease classes are tangled with source (Checkpoint 4), a module's correlation with "JDM" could be the lab. The column `r_JDM_vs_HC_within_jdm_lab` correlates the eigengene with JDM **only among JDM-lab samples** (JDM vs HC, both from the same lab). Here, batch cannot explain a difference. It's the cleanest evidence in the notebook.
- **`source_flag`.** `max_abs_r_source` is the largest |r| over the three source traits; above `BATCH_FLAG_R` the module is flagged as possibly batch-driven. `max_abs_r_disease` is the same for disease traits, so you can compare the two at a glance.
- **`n_ifn_genes`** counts how many canonical interferon genes (Part 1) fall in each module, which identifies the interferon module without guessing from the colour.

<div class="alert alert-info">
<b>How to read the summary.</b> A good "real biology" module has a high disease correlation, a <b>low</b> source correlation, <i>and</i> a clear within-lab JDM-vs-HC value. A module with high disease <i>and</i> high source correlation is ambiguous, and the within-lab column is the tiebreaker.
</div>

In [ ]:
# ---- module-trait table (per-trait sample size, Student p-values) -----------
n_per_trait <- colSums(!is.na(traits))
mtc <- cor(MEs, traits, use = "pairwise.complete.obs")
mtp <- mtc
for (j in seq_len(ncol(traits))) {
  n <- n_per_trait[j]
  t <- mtc[, j] * sqrt((n - 2) / pmax(1 - mtc[, j]^2, 1e-12))
  mtp[, j] <- 2 * pt(-abs(t), n - 2)
}
write.csv(mtc, file.path(OUT, "wgcna_module_trait_cor.csv"))
write.csv(mtp, file.path(OUT, "wgcna_module_trait_p.csv"))

# the one contrast that is NOT confounded with source: JDM vs HC inside the JDM-lab data
idx <- src3 == "jdm_lab" & !is.na(meta$class) & meta$class %in% c("HC", "JDM")
within_jdm <- as.numeric(cor(MEs[idx, , drop = FALSE], traits$JDM[idx]))
cat("\nJDM-lab samples used for the within-dataset JDM vs HC contrast:", sum(idx), "\n")

modules <- sub("^ME", "", colnames(MEs))
summ <- data.frame(
  module = modules,
  n_genes = as.integer(table(moduleColors)[modules]),
  n_ifn_genes = sapply(modules, function(m) sum(colnames(datExpr)[moduleColors == m] %in% IFN_GENES)),
  max_abs_r_source = apply(abs(mtc[, BATCH_TRAITS, drop = FALSE]), 1, max),
  max_abs_r_disease = apply(abs(mtc[, DISEASE_TRAITS, drop = FALSE]), 1, max),
  r_JDM_vs_HC_within_jdm_lab = round(within_jdm, 3))
summ$source_flag <- summ$max_abs_r_source > BATCH_FLAG_R
summ[, 4:5] <- round(summ[, 4:5], 3)
write.csv(summ, file.path(OUT, "wgcna_module_summary.csv"), row.names = FALSE)
cat("\nmodule summary:\n"); print(summ, row.names = FALSE)

---
# Part 9 · Gene tables, eigengene file, and the heatmap

🎯 **Goal:** save the gene-level results and a one-picture summary of Part 8.

💡 **Why:**

- **kME (module membership)** is each gene's correlation with its *own* module's eigengene. High kME marks a **hub gene**, the most representative member of the module. Sorting by kME is how you identify what a module "is about" (used in Part 11).
- **`is_IFN_gene`** is kept per gene so you can filter the table directly.
- The **eigengene file** carries `group` and `dataset` alongside each ME, so you can make your own boxplots by group or by lab without re-joining tables.
- The **heatmap** shows correlation (colour) and p-value (in brackets) for every module × trait pair. Red is positive and blue is negative.

In [ ]:
# ---- gene -> module, with module membership (kME) ---------------------------
kME <- cor(datExpr, MEs, use = "pairwise.complete.obs")
own <- paste0("ME", moduleColors)
gene_tbl <- data.frame(gene = colnames(datExpr), module = moduleColors,
                       kME_own_module = ifelse(own %in% colnames(kME),
                                               kME[cbind(seq_len(ncol(datExpr)),
                                                         match(own, colnames(kME)))], NA),
                       is_IFN_gene = colnames(datExpr) %in% IFN_GENES)
write.csv(gene_tbl, file.path(OUT, "wgcna_gene_modules.csv"), row.names = FALSE)
me_out <- cbind(sample_id = rownames(MEs), group = meta$group, dataset = meta$dataset, MEs)
write.csv(me_out, file.path(OUT, "wgcna_module_eigengenes.csv"), row.names = FALSE)

# ---- heatmap ----------------------------------------------------------------
png(file.path(OUT, "wgcna_module_trait_heatmap.png"), width = 1500, height = 300 + 55 * nrow(mtc), res = 120)
par(mar = c(8, 9, 3, 2))
txt <- matrix(paste0(signif(mtc, 2), "\n(", signif(mtp, 1), ")"), nrow = nrow(mtc))
labeledHeatmap(Matrix = mtc, xLabels = colnames(traits), yLabels = rownames(mtc),
               ySymbols = rownames(mtc), colorLabels = FALSE, colors = blueWhiteRed(50),
               textMatrix = txt, setStdMargins = FALSE, cex.text = 0.55, zlim = c(-1, 1),
               main = "Module-trait relationships")
dev.off()

cat("\nwritten to", OUT, ":\n")
for (f in sort(list.files(OUT))) cat(sprintf("  %-40s %6.2f MB\n", f, file.size(file.path(OUT, f)) / 1e6))

### 👀 Checkpoint 9: module–trait heatmap

**How to read it.** Each row is a module and each column a trait. Compare the three `src_*` columns with `HC / JDM / Interferonopathy`. A module that lights up the same way for a disease column *and* a source column is confounded. Cross-reference with `source_flag` and the within-lab column in the summary above.

In [ ]:
show_png(file.path(OUT, "wgcna_module_trait_heatmap.png"), width = 1000)

---
# Part 10 · Environment check

🎯 **Goal:** confirm which packages and tools are installed, and that the input files exist.

💡 **Why:** an R install on a managed JupyterHub often fails *silently* for compiled packages. This is a quick diagnostic you can paste into a support request. It also checks for conda/mamba, which is the usual route to fixing a missing system library, and confirms the two input files are visible from this session.

In [ ]:
for (p in c("png", "KEGGREST", "AnnotationDbi", "GO.db", "impute", "preprocessCore", "WGCNA"))
  cat(sprintf("%-15s installed: %s\n", p, requireNamespace(p, quietly = TRUE)))
cat("\nR home:", R.home(), "\n")
print(Sys.which(c("mamba", "conda", "micromamba")))
cat("\nexpression file exists:",
    file.exists("/mnt/file-systems/rnaseq_out/merged_log2_RPKM_QN_batchcorrected.csv"), "\n")
cat("samples file exists:   ",
    file.exists("/mnt/file-systems/rnaseq_out/merged_samples.csv"), "\n")

---
# Part 11 · Follow-up analyses: is each module biology or batch?

The WGCNA run gives you modules; these cells *interrogate* them. Each one answers a specific question, in this order:

| # | Question | Method |
|---|---|---|
| 11a | Which modules differ between JDM and HC **within the JDM lab** (no batch possible)? | per-module t-test, BH-adjusted |
| 11b | Is a module just tracking the **uncorrected** batch axis (PC1/PC2)? | eigengene–PC correlation |
| 11c | Does it *still* track PC1/PC2 **after** batch correction? | same, on the corrected matrix |
| 11d | What cell type is a module made of? | marker-gene counts per module |
| 11e | Are these genes even **expressed**? | mean-expression quantiles |
| 11f | Do whole **gene sets** (Hallmark/Reactome/modules) shift in JDM? | `mroast` |

<div class="alert alert-danger">
<b>Colour names are hard-coded below (<code>yellow</code>, <code>greenyellow</code>, <code>turquoise</code>, <code>blue</code>).</b> They came from an earlier run, and WGCNA assigns colours by module <i>size</i>. If you change <code>TOP_N</code>, the power, the merge cut, or the input data, the same colour can mean a <i>different</i> gene set. Check <code>summ</code> / <code>wgcna_gene_modules.csv</code> and update these names before trusting 11a, 11d and 11f.
</div>

<div class="alert alert-info">
Cells 11a–11f <b>re-read results from disk</b> and reassign <code>OUT</code>, <code>me</code>, <code>g</code>, <code>x</code> and <code>d</code>. They don't depend on the objects from Parts 1–9, so you can run Part 11 on its own after a restart. Just run <b>Part 1</b> first if you want <code>show_png</code>.
</div>

### 11a · JDM vs HC inside the JDM-lab data only

🎯 **Goal:** a clean, batch-free test of which modules separate JDM from healthy controls.

💡 **Why this subset:** both groups were run by the same lab, so a difference *cannot* be a dataset effect. Each module eigengene gets a Welch t-test (R's default `t.test`, which doesn't assume equal variances), and p-values are **Benjamini–Hochberg adjusted** for testing many modules at once. The second block lists the top hub genes (by kME) of the two modules of interest, to say what they are.

In [ ]:
OUT <- "/mnt/file-systems/rnaseq_out/wgcna"
me  <- read.csv(file.path(OUT, "wgcna_module_eigengenes.csv"), check.names = FALSE)
w   <- me[me$dataset == "jdm_lab" & me$group %in% c("HC", "JDM"), ]
mods <- grep("^ME", colnames(me), value = TRUE)
res <- do.call(rbind, lapply(mods, function(m) data.frame(
  module = sub("^ME", "", m),
  mean_HC  = round(mean(w[[m]][w$group == "HC"]), 3),
  mean_JDM = round(mean(w[[m]][w$group == "JDM"]), 3),
  p = t.test(w[[m]] ~ w$group)$p.value)))
res$p_BH <- p.adjust(res$p, "BH")
cat("JDM vs HC inside the JDM-lab data only (n =", nrow(w), "):\n")
print(res[order(res$p), ], row.names = FALSE, digits = 3)

g <- read.csv(file.path(OUT, "wgcna_gene_modules.csv"))
for (m in c("greenyellow", "yellow")) {
  x <- g[g$module == m, ]; x <- x[order(-x$kME_own_module), ]
  cat("\n", m, "-", nrow(x), "genes; top 15 by module membership:\n")
  print(head(x$gene, 15))
}

### 👀 Checkpoint 11a: eigengenes by group, within the JDM lab

In [ ]:
mm <- res$module[order(res$p)][1:min(4, nrow(res))]
op <- par(mfrow = c(1, length(mm)), mar = c(4, 4, 3, 1))
for (m in mm) boxplot(w[[paste0("ME", m)]] ~ w$group, main = paste0("ME", m),
                      ylab = "eigengene", xlab = "", col = c("#9ecae1", "#fc9272"))
par(op)

### 11b · Does a module track the *uncorrected* batch axis?

🎯 **Goal:** correlate every module eigengene with PC1 and PC2 of the **uncorrected** merged PCA (saved in `merged_samples.csv` by the Python run).

💡 **Why:** in the uncorrected data, the main axis of variation (PC1) was largely the dataset/lab. A module with |r| near 1 against PC1 is essentially *measuring the batch effect*. This tells you which modules the batch correction needed to rescue, and which were never batch-driven to begin with.

In [ ]:
OUT <- "/mnt/file-systems/rnaseq_out/wgcna"
me <- read.csv(file.path(OUT, "wgcna_module_eigengenes.csv"), check.names = FALSE)
ms <- read.csv("/mnt/file-systems/rnaseq_out/merged_samples.csv")
d  <- merge(me, ms[, c("sample_id", "PC1", "PC2")], by = "sample_id")
mods <- grep("^ME", colnames(d), value = TRUE)

cat("correlation of each module eigengene with PC1 / PC2 (uncorrected merged PCA):\n")
print(round(t(sapply(mods, function(m) c(PC1 = cor(d[[m]], d$PC1), PC2 = cor(d[[m]], d$PC2)))), 2))

cat("\nmean eigengene by dataset and group (IFN module, neutrophil module):\n")
print(aggregate(d[, c("MEyellow", "MEgreenyellow")],
                by = list(dataset = d$dataset, group = d$group),
                FUN = function(x) round(mean(x), 3)))

### 11c · Does it *still* track PC1/PC2 after batch correction?

🎯 **Goal:** redo 11b, but against a PCA recomputed from the **batch-corrected** matrix.

💡 **Why:** the WGCNA ran on the corrected matrix, but the PCs saved in `merged_samples.csv` came from the *uncorrected* one. To see whether any batch signal survives, recompute PCA on the corrected expression (top 2,000 variable genes, centred, not scaled, matching the Python PCA) and re-correlate. **If a module still correlates strongly with the new PC1, residual structure remains**. That structure could be real biology (e.g. disease) or leftover batch, so compare against 11a.

In [ ]:
OUT <- "/mnt/file-systems/rnaseq_out"
x  <- read.csv(file.path(OUT, "merged_log2_RPKM_QN_batchcorrected.csv"), row.names = 1, check.names = FALSE)
me <- read.csv(file.path(OUT, "wgcna", "wgcna_module_eigengenes.csv"), check.names = FALSE)

top <- order(apply(x, 1, var), decreasing = TRUE)[1:2000]
pc  <- prcomp(t(x[top, ]), center = TRUE, scale. = FALSE)
cat("variance explained, corrected matrix (PC1-3, %):",
    round(100 * pc$sdev[1:3]^2 / sum(pc$sdev^2), 1), "\n\n")

p <- data.frame(sample_id = rownames(pc$x), PC1c = pc$x[, 1], PC2c = pc$x[, 2])
d <- merge(me, p, by = "sample_id")
mods <- grep("^ME", colnames(d), value = TRUE)
cat("correlation of each module eigengene with the CORRECTED PC1 / PC2:\n")
print(round(t(sapply(mods, function(m)
  c(PC1c = cor(d[[m]], d$PC1c), PC2c = cor(d[[m]], d$PC2c)))), 2))

### 11d · What cell type is a module made of?

🎯 **Goal:** identify modules that are really a **blood cell composition** signal.

💡 **Why this matters for whole blood:** whole-blood RNA is a mixture of cell types, and their proportions differ between people and between labs (different handling, different globin depletion). A module made of haemoglobin genes isn't "disease"; it's red-cell content. A module full of `S100A8/S100A9/CSF3R` is neutrophil abundance. This cell lists the top 25 hub genes for two modules, then counts how many genes from each **marker panel** (erythroid/globin, platelet, T/NK, B cell, neutrophil/monocyte) land in each module.

**How to read the table:** rows are modules, columns are cell types. A module that concentrates one column (e.g. 10 of 14 erythroid markers) is a cell-composition module. Marker lists are short and hand-chosen, so treat counts as a strong hint, not proof.

In [ ]:
OUT <- "/mnt/file-systems/rnaseq_out/wgcna"
g <- read.csv(file.path(OUT, "wgcna_gene_modules.csv"))

for (m in c("turquoise", "blue")) {
  x <- g[g$module == m, ]; x <- x[order(-x$kME_own_module), ]
  cat("\n", m, "-", nrow(x), "genes; top 25 by module membership:\n")
  print(head(x$gene, 25))
}

markers <- list(
  erythroid_globin = c("HBA1","HBA2","HBB","HBD","HBM","ALAS2","SLC4A1","GYPA","GYPB","GYPC","CA1","AHSP","EPB42","BPGM"),
  platelet         = c("PPBP","PF4","GP9","ITGA2B","TUBB1","GNG11","SDPR"),
  T_NK             = c("CD3D","CD3E","CD2","IL7R","TRAC","LCK","CD247","NKG7","GZMA","GZMB","KLRB1"),
  B_cell           = c("CD79A","CD79B","MS4A1","CD19","IGKC","BANK1"),
  neutro_mono      = c("S100A8","S100A9","S100A12","FCGR3B","CXCR2","CSF3R","LYZ","CD14"))
tab <- sapply(markers, function(mk) tapply(g$gene %in% mk, g$module, sum))
cat("\nmarker genes found per module (marker list sizes:",
    paste(names(markers), lengths(markers), collapse = ", "), "):\n")
print(tab)

### 👀 Checkpoint 11d: cell-type marker enrichment per module

In [ ]:
if (is.matrix(tab) && nrow(tab) > 0) {
  op <- par(mar = c(5, 8, 3, 8), xpd = TRUE)
  barplot(t(tab), horiz = TRUE, las = 1, legend.text = colnames(tab),
          args.legend = list(x = "right", inset = c(-0.35, 0), cex = .7),
          main = "Cell-type marker genes per module", xlab = "marker genes found")
  par(op)
}

### 11e · Are these genes actually expressed?

🎯 **Goal:** check whether the network is built on genuinely expressed genes or on near-noise.

💡 **Why:** on the *uncorrected* log₂(RPKM+1) matrix, a gene with mean < 1 has RPKM below roughly 1. Genes that low are barely detected, and their "co-expression" can be a measurement artifact (quantile normalisation amplifies noise in low genes). The quantiles show the spread, and the final number is the **share of network genes with mean < 1**. A large share means the top-variance filter let in noisy genes, and you might raise the expression floor before re-running WGCNA.

In [ ]:
x <- read.csv("/mnt/file-systems/rnaseq_out/merged_log2_RPKM_QN.csv", row.names = 1, check.names = FALSE)
g <- read.csv("/mnt/file-systems/rnaseq_out/wgcna/wgcna_gene_modules.csv")
mu <- rowMeans(x[g$gene, ])
cat("genes in the WGCNA run:", nrow(g), "\nmean log2(RPKM+1) quantiles:\n")
print(round(quantile(mu, c(0, .05, .25, .5, .75, 1)), 2))
cat("share with mean log2(RPKM+1) < 1 (RPKM below about 1):", round(mean(mu < 1), 3), "\n")

### 11f · Gene-set testing with `mroast` (limma)

🎯 **Goal:** test whether *whole pathways* shift between JDM and HC, an independent check on what the modules suggest.

💡 **Why this approach:**

- **Same unconfounded subset:** JDM vs HC inside the JDM-lab data, so batch can't explain the result. It uses the **uncorrected** quantile-normalised matrix, which is fine here because there is only one dataset in the model.
- **Why `mroast`, not a per-gene test:** with only ~dozens of samples, per-gene tests are underpowered, but a *set* test pools evidence across all genes in a pathway. `mroast` tests whether the set as a whole moves, using **rotations** (a self-contained test that respects gene–gene correlation and doesn't need a permutation of the few samples). `NROT` sets p-value resolution.
- **Gene sets:** the 50 Hallmark sets plus three specific Reactome sets (neutrophil degranulation, IFN α/β signalling, IFN signalling), downloaded through `msigdbr`. The code handles both new and old `msigdbr` argument names and installs the `msigdbdf` data package if required.
- **`INCLUDE_MODULES`:** also tests your WGCNA modules as gene sets. This is a **sanity check, not independent evidence**: the modules were built from the same data, so a module scoring highly here is partly circular.
- **How to read the output:** `Direction` Up = higher in JDM than HC. `PropUp/PropDown` is the share of genes moving each way. `FDR` corrects for the number of sets tested. `PValue.Mixed` tests for movement in *either* direction, so a set can be significant mixed but not directional.

The rows to look for (the `key` list) are the interferon α/γ response sets, neutrophil degranulation, and the two WGCNA modules from 11a.

In [ ]:
suppressPackageStartupMessages(library(limma))
DIR <- "/mnt/file-systems/rnaseq_out"
OUT <- file.path(DIR, "mroast"); dir.create(OUT, showWarnings = FALSE, recursive = TRUE)
NROT <- 9999            # rotations; more = finer p-values
MIN_SET <- 10           # smallest gene set tested (genes present in the data)
INCLUDE_MODULES <- TRUE # also test your WGCNA modules (sanity check, not independent)
set.seed(1)

# ---- data: JDM vs HC inside the JDM-lab dataset only (the unconfounded contrast) ----
x  <- read.csv(file.path(DIR, "merged_log2_RPKM_QN.csv"), row.names = 1, check.names = FALSE)
ms <- read.csv(file.path(DIR, "merged_samples.csv"), check.names = FALSE, stringsAsFactors = FALSE)
ms <- ms[ms$dataset == "jdm_lab" & ms$group %in% c("HC", "JDM") & ms$sample_id %in% colnames(x), ]
y   <- as.matrix(x[, ms$sample_id])
grp <- factor(ms$group, levels = c("HC", "JDM"))
cat("samples used:\n"); print(table(grp))
design <- model.matrix(~ grp)                  # coefficient 2 = JDM vs HC

# ---- gene sets: Hallmark + a few Reactome sets ----
get_sets <- function() {
  if (!requireNamespace("msigdbr", quietly = TRUE)) stop("msigdbr is not installed")
  try_get <- function(...) tryCatch(msigdbr::msigdbr(...), error = function(e) e)
  h <- try_get(collection = "H")
  if (inherits(h, "error") && grepl("msigdbdf", conditionMessage(h))) {
    message("installing msigdbdf (gene-set data package) ...")
    install.packages("msigdbdf", repos = c("https://igordot.r-universe.dev", "https://cloud.r-project.org"))
    h <- try_get(collection = "H")
  }
  if (inherits(h, "error")) h <- try_get(category = "H")                        # older msigdbr
  if (inherits(h, "error")) stop(conditionMessage(h))
  r <- try_get(collection = "C2", subcollection = "CP:REACTOME")
  if (inherits(r, "error")) r <- try_get(category = "C2", subcategory = "CP:REACTOME")
  if (inherits(r, "error")) stop(conditionMessage(r))
  r <- r[r$gs_name %in% c("REACTOME_NEUTROPHIL_DEGRANULATION",
                          "REACTOME_INTERFERON_ALPHA_BETA_SIGNALING",
                          "REACTOME_INTERFERON_SIGNALING"), ]
  d <- rbind(as.data.frame(h)[, c("gs_name", "gene_symbol")],
             as.data.frame(r)[, c("gs_name", "gene_symbol")])
  split(d$gene_symbol, d$gs_name)
}
sets <- get_sets()
if (INCLUDE_MODULES) {
  g <- read.csv(file.path(DIR, "wgcna", "wgcna_gene_modules.csv"))
  g <- g[g$module != "grey", ]
  sets <- c(sets, split(g$gene, paste0("WGCNA_", g$module)))
}
idx <- lapply(sets, function(s) which(rownames(y) %in% unique(s)))
idx <- idx[lengths(idx) >= MIN_SET]
cat("gene sets tested:", length(idx), "\n")

# ---- mroast ----
res <- mroast(y, index = idx, design = design, contrast = 2, nrot = NROT)
res <- cbind(set = rownames(res), res)
write.csv(res, file.path(OUT, "mroast_JDM_vs_HC_jdm_lab.csv"), row.names = FALSE)

cat("\ntop 15 sets (Direction Up = higher in JDM than HC):\n")
print(head(res[, c("set", "NGenes", "PropUp", "PropDown", "Direction", "PValue", "FDR")], 15),
      row.names = FALSE, digits = 3)
key <- c("HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_GAMMA_RESPONSE",
         "REACTOME_NEUTROPHIL_DEGRANULATION", "WGCNA_yellow", "WGCNA_greenyellow")
cat("\nthe sets we care about:\n")
print(res[res$set %in% key, c("set", "NGenes", "Direction", "PValue", "FDR", "PValue.Mixed")],
      row.names = FALSE, digits = 3)
cat("\nwritten to", OUT, "\n")

### 👀 Checkpoint 11f: the top gene sets

In [ ]:
top <- head(res[order(res$PValue), ], 15)
val <- -log10(top$FDR) * ifelse(top$Direction == "Up", 1, -1)   # signed, in table order
op <- par(mar = c(4, 22, 3, 2))
barplot(rev(val), horiz = TRUE, names.arg = rev(sub("^HALLMARK_|^REACTOME_", "", top$set)),
        las = 1, cex.names = .65, col = ifelse(rev(val) > 0, "#fc9272", "#9ecae1"),
        xlab = "signed -log10(FDR)   (right = higher in JDM)", main = "mroast: JDM vs HC (JDM lab only)")
abline(v = c(-log10(.05), log10(.05)), lty = 2)
par(op)

---
## ✅ Reading the whole analysis

A defensible conclusion needs **agreement across independent lines of evidence**:

1. **Part 8:** the module correlates with a disease trait, has a **low** source correlation, and a clear **within-lab** JDM-vs-HC value.
2. **11a:** it separates JDM from HC inside the JDM lab after BH correction.
3. **11b/11c:** it is *not* simply the batch axis (or you understand why it still tracks PC1 after correction).
4. **11d/11e:** its genes are real cell-type or pathway genes, not noise or red-cell contamination.
5. **11f:** the matching **pathway** (interferon response, neutrophil degranulation) shifts in the expected direction.

A module passing 1–3 but failing 4 is likely a **cell-composition** effect, which can be real biology but isn't a pathway-level finding. A module that fails 3 is probably residual batch.